# Étape 3 — Comparaison inter-pages + score de confiance

Importe `comparer_dossier` depuis `src/comparison.py`. La logique de
similarité (Levenshtein pour les noms, comparaison positionnelle tolérante
aux confusions OCR pour les dates/identifiants) est documentée dans ce fichier.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

import json
from pathlib import Path
from src.comparison import comparer_dossier

OCR_RESULTS_DIR = Path("../outputs/ocr_results")


def charger_ocr(dossier_json_path: Path) -> dict:
    with open(dossier_json_path, "r", encoding="utf-8") as f:
        return json.load(f)

## Test sur deux dossiers déjà validés

`Dossier_000001` doit ressortir cohérent ; `Dossier_000005` doit signaler
une anomalie sur `nom_prenom`, page 3.

In [2]:
for nom_fichier in ["Dossier_000001_ocr.json", "Dossier_000005_ocr.json"]:
    print(f"=== {nom_fichier} ===")
    dossier = charger_ocr(OCR_RESULTS_DIR / nom_fichier)
    rapport = comparer_dossier(dossier["pages"])

    print(f"Cohérent : {rapport['coherent']}")
    for champ, detail in rapport["details"].items():
        print(f"  {champ}: référence page {detail['page_reference']} = {detail['valeur_reference']!r}")
        for c in detail["comparaisons"]:
            statut = "OK" if c["coherent"] else "ANOMALIE"
            print(f"    page {c['page']}: {c['valeur']!r} (score={c['score_similarite']:.2f}) [{statut}]")

    if rapport["anomalies"]:
        print("  --> Anomalies détectées :")
        for a in rapport["anomalies"]:
            print(f"      {a['champ']} : page {a['page_reference']} ({a['valeur_reference']!r}) "
                  f"vs page {a['page_suspecte']} ({a['valeur_suspecte']!r}), score={a['score_similarite']:.2f}")
    print()

=== Dossier_000001_ocr.json ===
Cohérent : True
  numero_dossier: référence page 1 = 'DOS202600001'
    page 4: 'DOS202600001' (score=1.00) [OK]
  nom_prenom: référence page 1 = 'SBALYoussef'
    page 2: 'SBAl Youssef' (score=1.00) [OK]
    page 3: 'SBAI.Youssef' (score=0.83) [OK]

=== Dossier_000005_ocr.json ===
Cohérent : False
  nom_prenom: référence page 1 = 'FASSLElHRL Chaimae'
    page 2: 'FAssI FIHRI Chaimae' (score=0.76) [OK]
    page 3: 'OUAHBI Asmaa' (score=0.24) [ANOMALIE]
    page 4: 'FASSI FIHRI Chaimae' (score=0.76) [OK]
  date_naissance: référence page 1 = 'L14/11/1995'
    page 2: '14/11/1995' (score=0.91) [OK]
  --> Anomalies détectées :
      nom_prenom : page 1 ('FASSLElHRL Chaimae') vs page 3 ('OUAHBI Asmaa'), score=0.24



## À faire ensuite

Pour ajuster les seuils, modifie `SEUILS_CHAMPS` dans `src/comparison.py`.